# Summary Stooq

Health check + coverage report for Stooq data in the local DuckDB.

Stooq writes two tables: **`prices`** (daily OHLCV) and **`markets`** (ticker → market mapping).

Row-level retrieval goes through `irp.data.stooq.prices()` / `markets()`. Aggregations stay in SQL via the shared `db()` connection.

In [1]:
import pandas as pd
from IPython.display import display

from irp.data._common import db
from irp.data.stooq import prices, markets

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Tables

Schema, row count, and key stats for each table written by `irp.sources.stooq.StooqSource`.

### `prices`

Daily OHLCV bars. One row per (Ticker, Date). `Src='stooq'` is the only loader. Date is a `DATE` value (`YYYY-MM-DD`).

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase, suffix-stripped symbol (e.g. `AAPL`) |
| Date | DATE | Bar date (e.g. `2024-01-02`) |
| O / H / L / C | DOUBLE | Open / High / Low / Close (snapshot-adjusted by Stooq) |
| V | DOUBLE | Volume |
| SrcId | VARCHAR | Original Stooq ticker incl. suffix (e.g. `aapl.us`) |
| Src | VARCHAR | Loader name; always `stooq` here |

In [2]:
_sample_prices = prices(tickers='AAPL', start='2025-04-01')
display(_sample_prices.dtypes.to_frame('dtype'))
display(_sample_prices.tail())

,dtype
Ticker,str
Date,datetime64[us]
O,float64
H,float64
L,float64
C,float64
V,float64
SrcId,str
Src,str


,Ticker,Date,O,H,L,C,V,SrcId,Src
278,AAPL,2026-05-11,291.979,293.88,290.23,292.68,42247285.0,AAPL.US,stooq
279,AAPL,2026-05-12,292.560,295.27,292.56,294.80,45748129.0,AAPL.US,stooq
280,AAPL,2026-05-13,293.500,300.92,293.50,298.87,52684260.0,AAPL.US,stooq
281,AAPL,2026-05-14,299.820,300.45,295.38,298.21,35324922.0,AAPL.US,stooq
282,AAPL,2026-05-15,297.900,303.20,296.52,300.23,54862836.0,AAPL.US,stooq


In [3]:
_stats_prices = db().execute("""
    SELECT
        COUNT(*)                     AS rows,
        COUNT(DISTINCT Ticker)       AS tickers,
        COUNT(DISTINCT SrcId)        AS src_ids,
        MIN(Date)                    AS date_min,
        MAX(Date)                    AS date_max,
        COUNT(DISTINCT Date)         AS distinct_dates
    FROM prices
""").df().T
_stats_prices.columns = ['prices']
display(_stats_prices)

,prices
rows,45758693
tickers,14480
src_ids,14532
date_min,1789-05-01 00:00:00
date_max,2026-05-16 00:00:00
distinct_dates,45001


### `markets`

Ticker → market mapping. One row per (SrcId). `Market` is Stooq's category folder (e.g. `nasdaq stocks`, `nyse etfs`, `bonds`).

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase, suffix-stripped symbol |
| Market | VARCHAR | Stooq category folder (lowercase) |
| SrcId | VARCHAR | Original Stooq ticker incl. suffix |
| Src | VARCHAR | Always `stooq` |

In [4]:
_mk_all = markets()
display(_mk_all.dtypes.to_frame('dtype'))
display(_mk_all.tail())

,dtype
Ticker,str
Market,str
SrcId,str
Src,str


,Ticker,Market,SrcId,Src
14527,^_UK,stooq stocks indices,^_uk,stooq
14528,^_US,stooq stocks indices,^_us,stooq
14529,^_USNM,stooq stocks indices,^_usnm,stooq
14530,^_USNQ,stooq stocks indices,^_usnq,stooq
14531,^_USNS,stooq stocks indices,^_usns,stooq


In [5]:
_stats_markets = db().execute("""
    SELECT
        COUNT(*)               AS rows,
        COUNT(DISTINCT Ticker) AS tickers,
        COUNT(DISTINCT SrcId)  AS src_ids
    FROM markets
""").df().T
_stats_markets.columns = ['markets']
display(_stats_markets)

,markets
rows,14532
tickers,14480
src_ids,14532


In [6]:
_mk = (
    _mk_all.groupby('Market')
    .agg(tickers=('Ticker', 'nunique'), rows=('Ticker', 'count'))
    .sort_values('tickers', ascending=False)
    .reset_index()
)
print(f'Total markets : {len(_mk)}  |  Total tickers : {_mk["tickers"].sum():,}')
display(_mk)

Total markets : 11  |  Total tickers : 14,532


,Market,tickers,rows
0,nasdaq stocks,4643,4643
1,nyse stocks,3670,3670
2,nyse etfs,2552,2552
3,currencies,1822,1822
4,nasdaq etfs,938,938
5,nysemkt stocks,306,306
6,bonds,284,284
7,cryptocurrencies,215,215
8,indices,62,62
9,money market,27,27


## Cross-table coverage

In [7]:
display(db().execute("""
    SELECT
        COUNT(DISTINCT p.Ticker)                                          AS prices_tickers,
        COUNT(DISTINCT m.Ticker)                                          AS markets_tickers,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE m.Ticker IS NULL)          AS in_prices_only,
        COUNT(DISTINCT m.Ticker) FILTER (WHERE p.Ticker IS NULL)          AS in_markets_only,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NOT NULL)      AS prices_with_company,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NULL)          AS prices_without_company
    FROM prices p
    FULL OUTER JOIN markets m   ON p.Ticker = m.Ticker AND p.Src = m.Src
    LEFT JOIN      companies c  ON p.Ticker = c.Ticker
""").df().T.rename(columns={0: 'count'}))
print('Tickers in prices but not in companies: non-equity instruments (FX, indices, bonds, crypto) and equities not covered by SimFin.')

,count
prices_tickers,14480
markets_tickers,14480
in_prices_only,0
in_markets_only,0
prices_with_company,4107
prices_without_company,10373


Tickers in prices but not in companies: non-equity instruments (FX, indices, bonds, crypto) and equities not covered by SimFin.


## Freshness — when was data last updated?

Last bar date overall and per market. Recent expected for equity markets; older for delisted/discontinued instruments.

In [8]:
display(db().execute("""
    SELECT
        m.Market,
        MAX(p.Date)              AS latest_date,
        COUNT(DISTINCT p.Ticker) AS tickers,
        COUNT(*)                 AS rows
    FROM prices p
    LEFT JOIN markets m ON p.Ticker = m.Ticker AND p.Src = m.Src
    GROUP BY m.Market
    ORDER BY latest_date DESC
""").df())

,Market,latest_date,tickers,rows
0,nasdaq etfs,2026-05-16,938,1343528
1,nysemkt stocks,2026-05-16,306,847048
2,nasdaq stocks,2026-05-16,4643,9139267
3,cryptocurrencies,2026-05-16,215,633805
4,nyse stocks,2026-05-16,3670,10629192
5,nyse etfs,2026-05-16,2552,5850923
6,bonds,2026-05-15,284,1310620
7,money market,2026-05-15,27,138143
8,stooq stocks indices,2026-05-15,13,38458
9,currencies,2026-05-15,1822,15500000


## Quality

Quick checks. For full anomaly review use `notebooks/review_stooq_anomalies.ipynb`. For Stooq vs Yahoo divergence (snapshot staleness) see `notebooks/compare_stooq_yahoo.ipynb` and `data/data_quality/stooq_staleness.csv`.

In [9]:
display(db().execute("""
    SELECT
        COUNT(*) FILTER (WHERE O IS NULL) AS missing_O,
        COUNT(*) FILTER (WHERE H IS NULL) AS missing_H,
        COUNT(*) FILTER (WHERE L IS NULL) AS missing_L,
        COUNT(*) FILTER (WHERE C IS NULL) AS missing_C,
        COUNT(*) FILTER (WHERE V IS NULL) AS missing_V
    FROM prices
""").df())

,missing_O,missing_H,missing_L,missing_C,missing_V
0,0,0,0,0,0


In [10]:
_neg_tickers = db().execute(
    'SELECT DISTINCT Ticker FROM prices WHERE C<0 OR O<0 OR H<0 OR L<0'
).df()['Ticker'].tolist()
if _neg_tickers:
    _neg = prices(tickers=_neg_tickers)
    _neg = _neg[(_neg['C'] < 0) | (_neg['O'] < 0) | (_neg['H'] < 0) | (_neg['L'] < 0)]
    _neg = _neg.merge(_mk_all[['Ticker', 'Market']], on='Ticker', how='left')
    print(f'Rows with negative prices : {len(_neg):,}  |  Tickers : {len(_neg_tickers)}')
    print('Expected for bonds and money-market instruments (yields can go negative).')
    display(_neg.groupby('Market', dropna=False).agg(tickers=('Ticker', 'nunique'), rows=('Ticker', 'count')).sort_values('tickers', ascending=False))
else:
    print('No negative-price rows.')

Rows with negative prices : 128,545  |  Tickers : 121
Expected for bonds and money-market instruments (yields can go negative).


,tickers,rows
Market,,
bonds,113,127700
money market,8,845


In [11]:
display(db().execute("""
    SELECT p.Ticker, m.Market, COUNT(*) AS rows
    FROM prices p
    LEFT JOIN markets m ON p.Ticker = m.Ticker AND p.Src = m.Src
    WHERE p.H < p.L OR p.C > p.H OR p.C < p.L
    GROUP BY p.Ticker, m.Market
    ORDER BY rows DESC
    LIMIT 20
""").df())

,Ticker,Market,rows
0,1YCHY,bonds,587
1,^PSEI,indices,224
2,^AOR,indices,130
3,^TWSE,indices,71
4,^JCI,indices,58
5,^KLCI,indices,33
6,^SHBS,indices,22
7,^OMXR,indices,14
8,^HEX,indices,12
9,^OMXV,indices,11
